# **Soluciones al problema de la predicción**

# Notebook 02 — Predicción numérica (regresión) con retorno semanal

En este notebook se reformula el problema para predecir un valor numérico más estable que el precio absoluto.  
Se define como objetivo el retorno a un horizonte semanal \(H=5\) días, y se construyen variables derivadas (*feature engineering*) a partir de OHLCV para alimentar modelos de regresión.

El rendimiento se evaluará estrictamente sobre un conjunto de prueba temporalmente posterior, comparando siempre frente a baselines.

In [3]:
import numpy as np
import pandas as pd

# Ajusta el path si tu csv está en otra ruta
df = pd.read_csv("/content/AAPL_processed.csv", parse_dates=["date"])
df = df.sort_values("date").reset_index(drop=True)

print("Shape:", df.shape)
print("Rango de fechas:", df["date"].min(), "→", df["date"].max())
print("Nulos por columna:\n", df.isna().sum())

df.head()

Shape: (11366, 6)
Rango de fechas: 1980-12-12 00:00:00 → 2026-01-16 00:00:00
Nulos por columna:
 date      0
open      0
high      0
low       0
close     0
volume    0
dtype: int64


,date,open,high,low,close,volume
0,1980-12-12,0.098389,0.098817,0.098389,0.098389,469033600
1,1980-12-15,0.093684,0.093684,0.093256,0.093256,175884800
2,1980-12-16,0.086839,0.086839,0.086412,0.086412,105728000
3,1980-12-17,0.088550,0.088978,0.088550,0.088550,86441600
4,1980-12-18,0.091118,0.091545,0.091118,0.091118,73449600


Se define el retorno a horizonte \(H\) como:

$$
r_{t,H} = \frac{P_{t+H} - P_t}{P_t}
$$

donde \(P_t\) es el precio de cierre en el instante \(t\).  
En este notebook se utiliza \(H=5\) (aproximadamente una semana bursátil).

In [4]:
H = 5  # horizonte semanal (5 días de trading)
P = df["close"].values

# r_{t,H} = (P_{t+H} - P_t) / P_t
y = (P[H:] - P[:-H]) / (P[:-H] + 1e-12)

# Para alinear X con y, nos quedamos con las filas hasta N-H
df_feat = df.iloc[:-H].copy()
df_feat["y_return_H"] = y

print("df_feat shape:", df_feat.shape)
df_feat[["date", "close", "y_return_H"]].head(10)


df_feat shape: (11361, 7)


,date,close,y_return_H
0,1980-12-12,0.098389,-0.017390
1,1980-12-15,0.093256,0.087151
2,1980-12-16,0.086412,0.222777
3,1980-12-17,0.088550,0.256041
4,1980-12-18,0.091118,0.333328
5,1980-12-19,0.096678,0.274335
6,1980-12-22,0.101384,0.185658
7,1980-12-23,0.105662,0.105263
8,1980-12-24,0.111223,0.061541
9,1980-12-26,0.121490,-0.049292


## **Split temporal**

La partición del conjunto se realiza respetando el orden temporal para evitar *data leakage*.  
Se emplea una división 70/15/15 sobre las observaciones alineadas con el target.

In [5]:
N = len(df_feat)

train_end = int(0.70 * N)
val_end   = int(0.85 * N)

train_df = df_feat.iloc[:train_end].copy()
val_df   = df_feat.iloc[train_end:val_end].copy()
test_df  = df_feat.iloc[val_end:].copy()

print("Train:", train_df.shape, train_df["date"].min(), "→", train_df["date"].max())
print("Val:  ", val_df.shape,   val_df["date"].min(),   "→", val_df["date"].max())
print("Test: ", test_df.shape,  test_df["date"].min(),  "→", test_df["date"].max())

Train: (7952, 7) 1980-12-12 00:00:00 → 2012-06-19 00:00:00
Val:   (1704, 7) 2012-06-20 00:00:00 → 2019-03-29 00:00:00
Test:  (1705, 7) 2019-04-01 00:00:00 → 2026-01-09 00:00:00


## **Returns diarios**: base del nuevo análisis

Se calcula el retorno diario como:

$$
r_t = \frac{P_t - P_{t-1}}{P_{t-1}}
$$

Este retorno se utiliza como señal básica para la construcción de variables derivadas.


In [6]:
# Retorno diario
df_feat["return_1"] = df_feat["close"].pct_change()

# Quitamos el primer NaN
df_feat = df_feat.dropna().reset_index(drop=True)

df_feat[["date", "close", "return_1", "y_return_H"]].head()

,date,close,return_1,y_return_H
0,1980-12-15,0.093256,-0.052171,0.087151
1,1980-12-16,0.086412,-0.073397,0.222777
2,1980-12-17,0.088550,0.024751,0.256041
3,1980-12-18,0.091118,0.028992,0.333328
4,1980-12-19,0.096678,0.061029,0.274335


## **Lags de return**

Se introducen retardos (*lags*) del retorno diario con el fin de capturar dependencias temporales de corto plazo.

In [7]:
LAGS = [1, 2, 3, 5, 10]

for lag in LAGS:
    df_feat[f"return_lag_{lag}"] = df_feat["return_1"].shift(lag)

df_feat = df_feat.dropna().reset_index(drop=True)
df_feat.filter(regex="return").head()

,y_return_H,return_1,return_lag_1,return_lag_2,return_lag_3,return_lag_5,return_lag_10
0,-0.120995,-0.024304,0.014084,0.092309,0.052628,0.048670,-0.052171
1,-0.113552,-0.028468,-0.024304,0.014084,0.092309,0.042199,-0.073397
2,-0.076088,0.010988,-0.028468,-0.024304,0.014084,0.052628,0.024751
3,-0.062965,-0.021737,0.010988,-0.028468,-0.024304,0.092309,0.028992
4,-0.054260,-0.044448,-0.021737,0.010988,-0.028468,0.014084,0.061029


## **Volatilidad rolling**

La volatilidad local se estima mediante la desviación estándar de los retornos diarios en ventanas temporales deslizantes.

In [8]:
VOL_WINDOWS = [5, 10, 20]

for w in VOL_WINDOWS:
    df_feat[f"volatility_{w}"] = df_feat["return_1"].rolling(window=w).std()

df_feat = df_feat.dropna().reset_index(drop=True)
df_feat.filter(regex="volatility").head()

,volatility_5,volatility_10,volatility_20
0,0.014330,0.025047,0.028758
1,0.015685,0.027570,0.029038
2,0.014320,0.029052,0.029409
3,0.018326,0.033156,0.030987
4,0.020073,0.026004,0.032716


## **Momentum**

Se incluyen variables de momentum que representan la variación acumulada del precio en distintos horizontes temporales.

In [9]:
MOM_WINDOWS = [5, 10, 20]

for m in MOM_WINDOWS:
    df_feat[f"momentum_{m}"] = df_feat["close"] - df_feat["close"].shift(m)

df_feat = df_feat.dropna().reset_index(drop=True)
df_feat.filter(regex="momentum").head()

,momentum_5,momentum_10,momentum_20
0,-0.006845,-0.006845,-0.023100
1,0.000000,-0.002567,-0.018395
2,0.007700,0.001284,-0.011550
3,0.006845,0.003851,-0.005561
4,0.008556,0.000429,-0.001283


## **Volumen transformado (no raw)**

El volumen se transforma mediante estadísticas móviles con el fin de capturar desviaciones respecto a su comportamiento reciente.

In [10]:
VOLM_WINDOWS = [5, 10, 20]

for w in VOLM_WINDOWS:
    df_feat[f"volume_mean_{w}"] = df_feat["volume"].rolling(w).mean()
    df_feat[f"volume_std_{w}"]  = df_feat["volume"].rolling(w).std()

df_feat = df_feat.dropna().reset_index(drop=True)

El conjunto final de características se construye a partir de retornos retardados, estimaciones de volatilidad, variables de momentum y estadísticas del volumen.

In [11]:
FEATURE_COLS = [
    c for c in df_feat.columns
    if c not in ["date", "open", "high", "low", "close", "volume", "y_return_H"]
]

X = df_feat[FEATURE_COLS]
y = df_feat["y_return_H"]

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (11292, 18)
y shape: (11292,)


## Rehacer split temporal con features

In [12]:
N = len(df_feat)

train_end = int(0.70 * N)
val_end   = int(0.85 * N)

X_train = X.iloc[:train_end]
y_train = y.iloc[:train_end]

X_val = X.iloc[train_end:val_end]
y_val = y.iloc[train_end:val_end]

X_test = X.iloc[val_end:]
y_test = y.iloc[val_end:]

print("Train:", X_train.shape)
print("Val:  ", X_val.shape)
print("Test: ", X_test.shape)

Train: (7904, 18)
Val:   (1694, 18)
Test:  (1694, 18)


## **Baselines para regresión del retorno semanal**

Como baseline simple, se utiliza un modelo constante que predice el valor medio del retorno semanal observado en el conjunto de entrenamiento:

$$
\hat{y}_t = \bar{y}_{train}
$$

Este baseline sirve como referencia mínima para evaluar si el modelo aprende señal predictiva.

In [13]:
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

y_pred_const = np.full_like(y_test.values, fill_value=y_train.mean(), dtype=float)

mae_const = mean_absolute_error(y_test, y_pred_const)
rmse_const = np.sqrt(mean_squared_error(y_test, y_pred_const))

print("BASELINE CONSTANTE (media train) — Test")
print(f"MAE:  {mae_const:.6f}")
print(f"RMSE: {rmse_const:.6f}")

BASELINE CONSTANTE (media train) — Test
MAE:  0.030854
RMSE: 0.040830


Para modelos lineales, se estandarizan las características utilizando:

$$
x' = \frac{x - \mu}{\sigma}
$$

donde $\mu$ y $\sigma$ se estiman únicamente en el conjunto de entrenamiento para evitar *data leakage*.


In [14]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_val_s   = scaler.transform(X_val)
X_test_s  = scaler.transform(X_test)

### **Ridge regression**

Ridge Regression estima un modelo lineal con regularización $L_2$:

$$
\min_{\mathbf{w}, b} \sum_{i=1}^{N} (y_i - \mathbf{w}^\top x_i - b)^2 + \lambda \|\mathbf{w}\|_2^2
$$

El parámetro $\lambda$ se selecciona mediante validación.


In [15]:
from sklearn.linear_model import Ridge

alphas = [0.01, 0.1, 1.0, 10.0, 100.0]

best_alpha = None
best_val_rmse = float("inf")
best_model = None

for a in alphas:
    m = Ridge(alpha=a)
    m.fit(X_train_s, y_train)

    val_pred = m.predict(X_val_s)
    val_rmse = np.sqrt(mean_squared_error(y_val, val_pred))

    if val_rmse < best_val_rmse:
        best_val_rmse = val_rmse
        best_alpha = a
        best_model = m

print("Mejor alpha (validación):", best_alpha)
print("RMSE validación:", best_val_rmse)

Mejor alpha (validación): 100.0
RMSE validación: 0.03672827913370003


In [16]:
y_pred_ridge = best_model.predict(X_test_s)

mae_ridge = mean_absolute_error(y_test, y_pred_ridge)
rmse_ridge = np.sqrt(mean_squared_error(y_test, y_pred_ridge))

print("\nRIDGE — Test")
print(f"MAE:  {mae_ridge:.6f}")
print(f"RMSE: {rmse_ridge:.6f}")


RIDGE — Test
MAE:  0.048768
RMSE: 0.062966


Los resultados muestran que el modelo Ridge no mejora al baseline constante, lo que indica que la relación entre las variables derivadas y el retorno semanal no puede ser capturada adecuadamente mediante un modelo lineal.

## **XGBoost**

In [17]:
!pip install xgboost

In [18]:
import xgboost as xgb

Se emplea un modelo de Gradient Boosting (XGBoost) para capturar relaciones no lineales entre las variables derivadas y el retorno semanal.  
Se utiliza una configuración conservadora para evitar sobreajuste.

In [21]:
print("xgboost version:", xgb.__version__)

model = xgb.XGBRegressor(
    n_estimators=2000,
    learning_rate=0.03,
    max_depth=4,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.0,
    reg_lambda=1.0,
    random_state=42,
    tree_method="hist",     # o "gpu_hist" si tienes GPU
    # 👇 Early stopping como callback en el CONSTRUCTOR
    callbacks=[xgb.callback.EarlyStopping(rounds=50, save_best=True)]
)

model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    verbose=False
)


xgboost version: 3.1.2


XGBRegressor(base_score=None, booster=None,
             callbacks=[<xgboost.callback.EarlyStopping object at 0x7c9df4c1cbc0>],
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=0.8, device=None, early_stopping_rounds=None,
             enable_categorical=False, eval_metric=None, feature_types=None,
             feature_weights=None, gamma=None, grow_policy=None,
             importance_type=None, interaction_constraints=None,
             learning_rate=0.03, max_bin=None, max_cat_threshold=None,
             max_cat_to_onehot=None, max_delta_step=None, max_depth=4,
             max_leaves=None, min_child_weight=None, missing=nan,
             monotone_constraints=None, multi_strategy=None, n_estimators=2000,
             n_jobs=None, num_parallel_tree=None, ...)

In [22]:
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Ejemplo: 70% train, 15% val, 15% test (temporal)
n = len(X)
train_end = int(n * 0.70)
val_end   = int(n * 0.85)

X_train, y_train = X.iloc[:train_end], y.iloc[:train_end]
X_val,   y_val   = X.iloc[train_end:val_end], y.iloc[train_end:val_end]
X_test,  y_test  = X.iloc[val_end:], y.iloc[val_end:]

print(X_train.shape, X_val.shape, X_test.shape)

(7904, 18) (1694, 18) (1694, 18)


In [25]:
model = xgb.XGBRegressor(
    n_estimators=3000,
    learning_rate=0.02,
    max_depth=4,
    min_child_weight=5,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.0,
    reg_lambda=1.0,
    random_state=42,
    tree_method="hist",
    objective="reg:squarederror",
    eval_metric="mae",
    callbacks=[
        xgb.callback.EarlyStopping(
            rounds=200,
            metric_name="mae",
            data_name="validation_0",
            save_best=True
        )
    ]
)

model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    verbose=200
)


[0]	validation_0-mae:0.02708
[199]	validation_0-mae:0.02835


XGBRegressor(base_score=None, booster=None,
             callbacks=[<xgboost.callback.EarlyStopping object at 0x7c9df4601eb0>],
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=0.8, device=None, early_stopping_rounds=None,
             enable_categorical=False, eval_metric='mae', feature_types=None,
             feature_weights=None, gamma=None, grow_policy=None,
             importance_type=None, interaction_constraints=None,
             learning_rate=0.02, max_bin=None, max_cat_threshold=None,
             max_cat_to_onehot=None, max_delta_step=None, max_depth=4,
             max_leaves=None, min_child_weight=5, missing=nan,
             monotone_constraints=None, multi_strategy=None, n_estimators=3000,
             n_jobs=None, num_parallel_tree=None, ...)

In [26]:
yhat_val  = model.predict(X_val)
yhat_test = model.predict(X_test)

regression_report("VAL",  y_val,  yhat_val)
regression_report("TEST", y_test, yhat_test)


VAL -> MAE: 0.027084 | RMSE: 0.035867 | R2: -0.0039
TEST -> MAE: 0.030847 | RMSE: 0.040797 | R2: 0.0016


## **Conclusión**

En este notebook se ha abordado el problema de la predicción financiera desde una perspectiva de regresión, utilizando como variable objetivo el retorno semanal (H = 5 días) y un conjunto de características derivadas basadas en información histórica: lags de retornos, volatilidad rolling, momentum y volumen. Este planteamiento se fundamenta en la hipótesis de que los retornos agregados a varios días podrían contener una señal más estable que los retornos diarios.

Se han evaluado distintos enfoques de modelado siguiendo una progresión lógica:

Un baseline constante, que alcanza un MAE ≈ 0.031.

Un modelo lineal (Ridge regression), que no consigue superar al baseline, lo que sugiere que la relación entre las variables explicativas y el retorno no es puramente lineal.

Un modelo no lineal más expresivo (XGBoost Regressor), que logra una mejora clara en validación (MAE ≈ 0.027), pero cuya capacidad de generalización es muy limitada en el conjunto de test (MAE ≈ 0.0308, prácticamente equivalente al baseline).

Además, el coeficiente de determinación obtenido en test (R² ≈ 0.0016) indica que el modelo apenas consigue explicar la variabilidad del retorno futuro. Este resultado es consistente con la evidencia empírica ampliamente documentada en la literatura financiera: la predicción puntual de retornos mediante modelos supervisados presenta una señal extremadamente débil y difícil de explotar de forma estable.

A partir de estos resultados, se concluye que el enfoque de regresión para predecir la magnitud exacta del retorno no resulta adecuado para este conjunto de datos y este horizonte temporal, al menos con el conjunto de características consideradas. Sin embargo, esto no implica necesariamente la ausencia total de señal predictiva, sino más bien que dicha señal puede manifestarse de forma más clara en términos direccionales (subida/bajada) que en términos de magnitud.

Por este motivo, en el siguiente notebook se reformula el problema como una tarea de clasificación binaria, donde el objetivo será predecir el signo del retorno futuro (positivo vs. negativo). Este cambio de formulación está alineado tanto con prácticas habituales en quantitative finance como con la literatura académica, y permite evaluar si existe capacidad predictiva útil desde una perspectiva más realista y aplicable a estrategias de inversión.